In [ ]:
#exp13
import tensorflow as tf
import matplotlib.pyplot as plt
import pandas as pd

from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense, Dropout

In [ ]:
VOCAB_SIZE = 10000
MAX_LENGTH = 200
EMBEDDING_DIM = 32
RNN_UNITS = 32

EPOCHS = 5
BATCH_SIZE = 128

In [ ]:
(x_train, y_train), (x_test, y_test) = imdb.load_data(
    num_words=VOCAB_SIZE
)

In [ ]:
x_train = pad_sequences(
    x_train,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

x_test = pad_sequences(
    x_test,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

In [ ]:
def create_model(model_type):

    model = Sequential()

    model.add(
        Embedding(
            input_dim=VOCAB_SIZE,
            output_dim=EMBEDDING_DIM,
            input_length=MAX_LENGTH
        )
    )

    if model_type == "RNN":
        model.add(SimpleRNN(RNN_UNITS))

    elif model_type == "LSTM":
        model.add(LSTM(RNN_UNITS))

    elif model_type == "GRU":
        model.add(GRU(RNN_UNITS))

    model.add(Dropout(0.2))

    model.add(Dense(1, activation="sigmoid"))

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:
model_types = ["RNN", "LSTM", "GRU"]

models = {}
histories = {}
results = {}

for model_type in model_types:

    print("\n==============================")
    print(model_type)
    print("==============================")

    model = create_model(model_type)

    history = model.fit(
        x_train,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=0.2,
        verbose=1
    )

    test_loss, test_accuracy = model.evaluate(
        x_test,
        y_test,
        verbose=0
    )

    models[model_type] = model
    histories[model_type] = history

    results[model_type] = [
        test_loss,
        test_accuracy
    ]

    print("Test Loss:", test_loss)
    print("Test Accuracy:", test_accuracy)

In [ ]:
results_df = pd.DataFrame(
    results,
    index=["Test Loss", "Test Accuracy"]
).T

print("\nPerformance Comparison")
print(results_df)


plt.figure(figsize=(10, 6))

for model_type in model_types:

    plt.plot(
        histories[model_type].history["accuracy"],
        marker="o",
        label=model_type
    )

plt.title("Training Accuracy Comparison")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
for model_type in model_types:

    plt.plot(
        histories[model_type].history["val_accuracy"],
        marker="o",
        label=model_type
    )

plt.title("Validation Accuracy Comparison")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()


plt.figure(figsize=(10, 6))

In [ ]:
plt.figure(figsize=(10, 6))

for model_type in model_types:

    plt.plot(
        histories[model_type].history["val_loss"],
        marker="o",
        label=model_type
    )

plt.title("Validation Loss Comparison")
plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
accuracy = [
    results[m][1]
    for m in model_types
]

plt.figure(figsize=(8, 5))

bars = plt.bar(
    model_types,
    accuracy,
    color=["steelblue", "orange", "green"]
)

plt.title("Test Accuracy Comparison")
plt.xlabel("Model")
plt.ylabel("Test Accuracy")
plt.ylim(0, 1)

for bar, value in zip(bars, accuracy):

    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.01,
        f"{value:.4f}",
        ha="center"
    )

plt.show()


loss = [
    results[m][0]
    for m in model_types
]

plt.figure(figsize=(8, 5))

bars = plt.bar(
    model_types,
    loss,
    color=["steelblue", "orange", "green"]
)

plt.title("Test Loss Comparison")
plt.xlabel("Model")
plt.ylabel("Test Loss")

for bar, value in zip(bars, loss):

    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.01,
        f"{value:.4f}",
        ha="center"
    )

plt.show()

In [ ]:
for model_type in model_types:

    print(
        model_type,
        "Parameters:",
        models[model_type].count_params()
    )